# Chapter 8 — Memory and Selective Recall

**Book alignment:** current Chapter 8 · internal demo `Stage 07`

The shared demo package calls this **Stage 07** internally. The notebook number follows the book chapter number; the internal stage number is one lower.

**Question this notebook isolates:** Can selected past information cross an explicit lifecycle and causally improve a later decision without overruling current evidence?


## Hypothesis

Memory earns its place only when selected past information crosses a controlled lifecycle and changes a later decision usefully. Retrieval alone is not success, and current evidence must outrank stale memory.


In [ ]:
from pathlib import Path
import sys

def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'demo' / 'agents-from-first-principles').exists():
            return candidate
    raise RuntimeError('Run this notebook from a checkout containing demo/agents-from-first-principles')

REPO_ROOT = find_repo_root(Path.cwd().resolve())
DEMO_ROOT = REPO_ROOT / 'demo' / 'agents-from-first-principles'
sys.path.insert(0, str(DEMO_ROOT))

from first_principles_agent.memory import (
    MemoryCycle, MemoryDiagnostics, MemoryOutcome, MemoryRecord,
    MemoryStatus, MemoryStore, ProspectiveMemory, RetrievalPolicy, WritePolicy,
    memory_induced_regret,
)


## Experiment 1 - writing is selective

Transient or unverified information should remain state/trace information rather than becoming persistent memory.


In [ ]:
store = MemoryStore()
write_policy = WritePolicy()

unverified = store.write(memory_id='guess', key='orders_api', value='/v1/orders', scope='project', provenance='model guess', policy=write_policy, verified=False)
transient = store.write(memory_id='stdout', key='last_stdout', value='1 failed', scope='run', provenance='current trace', policy=write_policy, reusable=False)
old = store.write(memory_id='v1', key='orders_api', value='/v1/orders', scope='project', provenance='verified old run', policy=write_policy)

assert unverified is None
assert transient is None
assert old is not None
assert [record.id for record in store.records] == ['v1']


## Experiment 2 - current evidence outranks stale memory

The old memory is retrieved because it is active and relevant, but conflicting current evidence prevents it from entering the effective decision context or being used.


In [ ]:
cycle = MemoryCycle()
current = cycle.decide(
    store, key='orders_api', scope='project',
    current_evidence={'orders_api': '/v2/orders'},
    fallback='discover endpoint',
)

assert current.choice == '/v2/orders'
assert current.retrieved_ids == ('v1',)
assert current.included_ids == ()
assert current.used_ids == ()
current


## Experiment 3 - supersession changes the next run

Once the current evidence has been verified and persisted, the stale record is superseded. A later run with no fresh evidence retrieves and uses only the replacement.


In [ ]:
fresh = store.write(
    memory_id='v2', key='orders_api', value='/v2/orders', scope='project',
    provenance='verified current request', policy=write_policy, supersedes='v1',
)
next_run = cycle.decide(store, key='orders_api', scope='project', current_evidence={}, fallback='discover endpoint')

assert fresh is not None
assert store.get('v1').status == MemoryStatus.SUPERSEDED
assert next_run.choice == '/v2/orders'
assert next_run.retrieved_ids == ('v2',)
assert next_run.included_ids == ('v2',)
assert next_run.used_ids == ('v2',)
next_run


## Experiment 4 - lifecycle eligibility comes before retrieval ranking

The finished chapter treats lifecycle state as part of the memory contract. `SUPERSEDED`, `EXPIRED`, `FORGOTTEN`, and `DELETED` records are not merely lower-scoring memories; they are **ineligible** for ordinary retrieval.

This is the important ordering rule: **filter by eligibility first, rank second**.


In [ ]:
lifecycle_store = MemoryStore()
active = lifecycle_store.write(
    memory_id='active', key='test_command', value='pytest -q', scope='project',
    provenance='verified run', policy=write_policy,
)
expired = lifecycle_store.write(
    memory_id='expired', key='test_command', value='pytest old', scope='project',
    provenance='old verified run', policy=write_policy,
)
forgotten = lifecycle_store.write(
    memory_id='forgotten', key='test_command', value='pytest noisy', scope='project',
    provenance='obsolete run', policy=write_policy,
)
deleted = lifecycle_store.write(
    memory_id='deleted', key='test_command', value='pytest removed', scope='project',
    provenance='invalidated run', policy=write_policy,
)

lifecycle_store.expire('expired')
lifecycle_store.forget('forgotten')
lifecycle_store.delete('deleted')

eligible = RetrievalPolicy().retrieve(lifecycle_store, key='test_command', scope='project')

assert active is not None
assert lifecycle_store.get('expired').status == MemoryStatus.EXPIRED
assert lifecycle_store.get('forgotten').status == MemoryStatus.FORGOTTEN
assert lifecycle_store.get('deleted').status == MemoryStatus.DELETED
assert tuple(record.id for record in eligible) == ('active',)

[(record.id, record.status.value) for record in lifecycle_store.records], tuple(record.id for record in eligible)


## Experiment 5 - prospective memory survives intervening activity


In [ ]:
reminder = ProspectiveMemory('full-suite-after-patch', 'tests_pass_after_patch', 'run_full_suite')
events = ['read_file', 'edit_parser', 'tests_pass_after_patch', 'tests_pass_after_patch']
fired = [action for event in events if (action := reminder.observe(event)) is not None]
assert fired == ['run_full_suite']
assert reminder.fired is True
fired


## Experiment 6 - memory can help or hurt

A verified remembered test command can reduce rediscovery work. A stale active endpoint memory can also anchor the decision and make a memory-enabled run fail where a no-memory fallback succeeds.


In [ ]:
helpful_store = MemoryStore([MemoryRecord('test-command', 'test_command', 'pytest -q', 'project', 'verified prior run')])
helpful = cycle.decide(helpful_store, key='test_command', scope='project', current_evidence={}, fallback='discover test command')
no_memory_steps = 3
verified_memory_steps = 1 if helpful.used_ids else 3

stale_store = MemoryStore([MemoryRecord('stale', 'orders_api', '/v1/orders', 'project', 'old run')])
stale = cycle.decide(stale_store, key='orders_api', scope='project', current_evidence={}, fallback='/v2/orders')
no_memory_outcome = MemoryOutcome(success=True, steps=2)
memory_outcome = MemoryOutcome(success=stale.choice == '/v2/orders', steps=1)
regret = memory_induced_regret(no_memory=no_memory_outcome, with_memory=memory_outcome)

assert helpful.choice == 'pytest -q'
assert verified_memory_steps < no_memory_steps
assert stale.used_ids == ('stale',)
assert regret is True


## Diagnostics

Retrieved, included, used, and helpful remain separate measurable events.


In [ ]:
diagnostics = MemoryDiagnostics(
    write_precision=1.0,
    retrieval_recall=1.0,
    context_inclusion_rate=1.0,
    decision_use_rate=1.0,
    outcome_delta_steps=no_memory_steps - verified_memory_steps,
    memory_induced_regret=regret,
)
assert diagnostics.outcome_delta_steps == 2
assert diagnostics.memory_induced_regret is True
diagnostics.as_dict()


## What was earned

Persistent memory is now a controlled causal mechanism rather than a synonym for stored text. The implementation distinguishes writing, lifecycle status, retrieval, context inclusion, decision use and outcome effect. Current evidence outranks stale memory; supersession changes future retrieval; prospective intentions survive intervening activity; and memory-induced regret is observable.

State, execution trace, persistent memory and model context remain separate objects. No trajectory search has been introduced.
